In [1]:
from pathlib import Path
import csv
import json
import shutil

# === STUDENT INFO — COPY YOURS HERE ===
LAST_NAME = "Viana"
STUDENT_ID = "TUPM-26-1953"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

# Rules from the handout
RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_STATUS_CLASS = (SEED_DIGIT % 4) + 2  # 2=2xx, 3=3xx, 4=4xx, 5=5xx
print(f"Processing: {RECORDS_TO_PROCESS} records")
print(f"Target Status Class: {TARGET_STATUS_CLASS}xx")

Processing: 1026 records
Target Status Class: 5xx


In [3]:
# === EXERCISE 1: WEB SERVER LOG ANALYSIS ===
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(".")
BACKUP_DIR = Path("../backup")

input_path = DATA_DIR / "weblog.csv"
filtered_csv = OUTPUT_DIR / f"{LAST_NAME}_weblog_filtered.csv"
summary_json = OUTPUT_DIR / f"{LAST_NAME}_weblog_summary.json"
backup_path = BACKUP_DIR / summary_json.name

# Counters
status_counts = {}
url_counts = {}
matching_records = []

with open(input_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    processed = 0
    
    for row in reader:
        processed += 1
        if processed > RECORDS_TO_PROCESS:
            break
        
        try:
            status = row["Status"].strip()
            if status.startswith(str(TARGET_STATUS_CLASS)):
                matching_records.append(row)
                status_counts[status] = status_counts.get(status, 0) + 1
                url_counts[row["URL"]] = url_counts.get(row["URL"], 0) + 1
        except (KeyError, IndexError):
            continue

# Top 5 URLs
top_urls = sorted(url_counts.items(), key=lambda x: x[1], reverse=True)[:5]

# Export filtered CSV
with open(filtered_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["IP", "Time", "URL", "Status"])
    writer.writeheader()
    writer.writerows(matching_records)

# Export JSON summary
summary = {
    "records_processed": RECORDS_TO_PROCESS,
    "target_status_class": f"{TARGET_STATUS_CLASS}xx",
    "records_in_target_class": len(matching_records),
    "count_per_status_code": status_counts,
    "five_most_requested_urls": [{"url": u, "count": c} for u, c in top_urls]
}
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

# Backup
shutil.copy2(summary_json, backup_path)

# Output for your report
print("=== EXERCISE 1 RESULTS ===")
print(f"Records processed: {RECORDS_TO_PROCESS}")
print(f"Target status class: {TARGET_STATUS_CLASS}xx")
print(f"Records in target class: {len(matching_records)}")
print(f"Count per status code: {status_counts}")
print("Five most requested URLs:")
for u, c in top_urls:
    print(f"  {u}: {c}")
print(f"Filtered CSV file: {filtered_csv.name}")
print(f"JSON summary file: {summary_json.name}")
print(f"Backup verification: {backup_path.exists()}")

=== EXERCISE 1 RESULTS ===
Records processed: 1026
Target status class: 5xx
Records in target class: 0
Count per status code: {}
Five most requested URLs:
Filtered CSV file: Viana_weblog_filtered.csv
JSON summary file: Viana_weblog_summary.json
Backup verification: True
